<a href="https://colab.research.google.com/github/nebi19/30DaysOfFlutter/blob/main/AI_Phishing_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("Everything is ready!")

Everything is ready!


In [ ]:
data = {
    "url": [
        "https://google.com",
        "https://github.com",
        "https://microsoft.com",
        "https://apple.com",
        "https://amazon.com",

        "http://secure-login-example.com",
        "http://verify-account-example.com",
        "http://paypal-login-example.com",
        "http://account-security-example.com",
        "http://update-password-example.com"
    ],

    "label": [
        0,
        0,
        0,
        0,
        0,

        1,
        1,
        1,
        1,
        1
    ]
}

df = pd.DataFrame(data)

df

,url,label
0,https://google.com,0
1,https://github.com,0
2,https://microsoft.com,0
3,https://apple.com,0
4,https://amazon.com,0
5,http://secure-login-example.com,1
6,http://verify-account-example.com,1
7,http://paypal-login-example.com,1
8,http://account-security-example.com,1
9,http://update-password-example.com,1


In [ ]:
import re
from urllib.parse import urlparse

def extract_features(url):

    parsed = urlparse(url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""

    features = {
        "url_length": len(url),

        "hostname_length": len(hostname),

        "path_length": len(path),

        "dot_count": url.count("."),

        "hyphen_count": url.count("-"),

        "slash_count": url.count("/"),

        "digit_count": sum(c.isdigit() for c in url),

        "has_https": int(parsed.scheme == "https"),

        "has_ip": int(
            bool(re.match(
                r"^\d+\.\d+\.\d+\.\d+$",
                hostname
            ))
        ),

        "has_at": int("@" in url),

        "has_login": int("login" in url.lower()),

        "has_verify": int("verify" in url.lower()),

        "has_account": int("account" in url.lower()),

        "has_password": int("password" in url.lower())
    }

    return features

In [ ]:
test_url = "http://secure-login-example.com/verify-account"

features = extract_features(test_url)

features

{'url_length': 46,
 'hostname_length': 24,
 'path_length': 15,
 'dot_count': 1,
 'hyphen_count': 3,
 'slash_count': 3,
 'digit_count': 0,
 'has_https': 0,
 'has_ip': 0,
 'has_at': 0,
 'has_login': 1,
 'has_verify': 1,
 'has_account': 1,
 'has_password': 0}

In [ ]:
X = pd.DataFrame(
    df["url"].apply(extract_features).tolist()
)

y = df["label"]

print("Features:")
display(X)

print("Labels:")
display(y)

Features:


,url_length,hostname_length,path_length,dot_count,hyphen_count,slash_count,digit_count,has_https,has_ip,has_at,has_login,has_verify,has_account,has_password
0,18,10,0,1,0,2,0,1,0,0,0,0,0,0
1,18,10,0,1,0,2,0,1,0,0,0,0,0,0
2,21,13,0,1,0,2,0,1,0,0,0,0,0,0
3,17,9,0,1,0,2,0,1,0,0,0,0,0,0
4,18,10,0,1,0,2,0,1,0,0,0,0,0,0
5,31,24,0,1,2,2,0,0,0,0,1,0,0,0
6,33,26,0,1,2,2,0,0,0,0,0,1,1,0
7,31,24,0,1,2,2,0,0,0,0,1,0,0,0
8,35,28,0,1,2,2,0,0,0,0,0,0,1,0
9,34,27,0,1,2,2,0,0,0,0,0,0,0,1


Labels:


,label
0,0
1,0
2,0
3,0
4,0
5,1
6,1
7,1
8,1
9,1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8
Testing samples: 2


In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [ ]:
predictions = model.predict(X_test)

print(predictions)
print(y_test.values)

[0 1]
[0 1]


In [ ]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [ ]:
def check_url(url):

    features = extract_features(url)

    X_new = pd.DataFrame([features])

    prediction = model.predict(X_new)[0]

    probability = model.predict_proba(X_new)[0]

    if prediction == 1:
        result = "🚨 PHISHING"
        confidence = probability[1]
    else:
        result = "✅ LEGITIMATE"
        confidence = probability[0]

    print("URL:")
    print(url)

    print()
    print("Result:", result)
    print("Confidence:", round(confidence * 100, 2), "%")

In [ ]:
check_url("http://secure-login-example.com/verify-account")

URL:
http://secure-login-example.com/verify-account

Result: 🚨 PHISHING
Confidence: 99.0 %


In [ ]:
check_url("https://google.com")

URL:
https://google.com

Result: ✅ LEGITIMATE
Confidence: 100.0 %


In [ ]:
original_url = "http://secure-login-example.com/verify-account"

check_url(original_url)

URL:
http://secure-login-example.com/verify-account

Result: 🚨 PHISHING
Confidence: 99.0 %


In [ ]:
variant_1 = original_url.replace("-", "")
variant_2 = original_url.replace("/verify-account", "/account")
variant_3 = original_url.replace("http://", "https://")

print(variant_1)
print(variant_2)
print(variant_3)

http://secureloginexample.com/verifyaccount
http://secure-login-example.com/account
https://secure-login-example.com/verify-account


In [ ]:
check_url(variant_1)
check_url(variant_2)
check_url(variant_3)

URL:
http://secureloginexample.com/verifyaccount

Result: 🚨 PHISHING
Confidence: 82.0 %
URL:
http://secure-login-example.com/account

Result: 🚨 PHISHING
Confidence: 99.0 %
URL:
https://secure-login-example.com/verify-account

Result: 🚨 PHISHING
Confidence: 79.0 %


In [ ]:
def generate_variants(url):

    variants = {}

    variants["original"] = url

    variants["remove_hyphens"] = url.replace("-", "")

    variants["shorter_path"] = url.replace(
        "/verify-account",
        "/account"
    )

    variants["https"] = url.replace(
        "http://",
        "https://"
    )

    return variants

In [ ]:
variants = generate_variants(original_url)

for name, url in variants.items():

    print("=" * 60)
    print(name)

    check_url(url)

original
URL:
http://secure-login-example.com/verify-account

Result: 🚨 PHISHING
Confidence: 99.0 %
remove_hyphens
URL:
http://secureloginexample.com/verifyaccount

Result: 🚨 PHISHING
Confidence: 82.0 %
shorter_path
URL:
http://secure-login-example.com/account

Result: 🚨 PHISHING
Confidence: 99.0 %
https
URL:
https://secure-login-example.com/verify-account

Result: 🚨 PHISHING
Confidence: 79.0 %
